Using the file CompiledCANONInfo.txt, use the OpenAI API to train a chatbot that can answer questions about Star Wars


In [1]:
import openai
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain.chains.question_answering import load_qa_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import GPT4AllEmbeddings, HuggingFaceEmbeddings 
from langchain_community.llms import GPT4All
from langchain.chains import RetrievalQA
from gpt4all import Embed4All

In [ ]:
"""Generate vector database"""
# Load the .env file
load_dotenv()
# Load the OpenAI API key from the .env file
openai.api_key = os.getenv("OPENAI_API_KEY")
# Load the .txt file using the UnstructuredTextLoader
print("Loading data...")
loader = UnstructuredFileLoader("CompiledALLInfo.txt")
data = loader.load()
print("Data loaded.")

persist_directory = "vectDBALLINFOHuggingFace"
# embedding = GPT4AllEmbeddings()#OpenAIEmbeddings() 
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
def generateVectorDB(data):
    print("Splitting document...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    documents = text_splitter.split_documents(data)
    print("Number of documents: ", len(documents))
    print("Documents split.")
    """TODO: GET AN EMBEDDING THAT'LL LET ME USE THE VECTOR STORES"""
    print("Generating vector database...")
    db = Chroma.from_documents(documents, embedding, persist_directory=persist_directory)
    db.persist()
    print("Vector database created.")
generateVectorDB(data)
# print(docs[0].page_content)

In [2]:
# Now, we can use the QA chain to answer questions
# TODO: Setup database retrieval to then use to query and generate an answer
def generateRetriever(allInfo=False):
    if allInfo:
        persist_directory = "vectDBALLINFOHuggingFace"
    else:
        persist_directory = "vectDB"
    embedding = GPT4AllEmbeddings()
    # model_name = "all-MiniLM-L6-v2.gguf2.f16.gguf"
    # gpt4all_kwargs = {'allow_download': 'True'}
    # embeddings = GPT4AllEmbeddings(
    #     model_name=model_name,
    #     gpt4all_kwargs=gpt4all_kwargs
    # )
    # embedding = Embed4All()
    db = Chroma(persist_directory=persist_directory,embedding_function=embedding)
    retriever = db.as_retriever(search_type="similarity")
    print("Retriever created.")
    return retriever

def generateQAChain(retriever=None):
    myLLM = GPT4All(model="models/mistral-7b-openorca.Q4_0.gguf", n_threads=8)
    print("LLM created.")
    qa = RetrievalQA.from_chain_type(llm=myLLM, chain_type="map_reduce", retriever=retriever, return_source_documents=True)
    print("QA chain created.")
    return qa

qa = generateQAChain(retriever=generateRetriever(allInfo=True))


Retriever created.
LLM created.
QA chain created.


In [5]:
query = "Who is captain rex"
result = qa({"query": query})
print(result['result'])

 Captain Rex is a character from Star Wars, specifically a clone trooper who went on to become a commander before shedding his designation after Order 66 was issued.
